In [7]:
import numpy as np
import librosa
import joblib
import tensorflow as tf
from pathlib import Path

# Konfigurasi
SR = 22050
DURATION = 2.0
N_MELS = 64
SPEC_TIME_FRAMES = 87
TARGET_VECTOR_DIM = 183

# Load model dan scaler
dnn_model = tf.keras.models.load_model('advanced_dnn_audio_model.keras')
cnn_model = tf.keras.models.load_model('advanced_cnn_audio_model.keras')
meta = joblib.load('audio_scaler_and_labels.pkl')
scaler = meta['scaler']
class_names = meta['class_names']

# Helper untuk pad/trim
def pad_or_trim(y, sr, duration):
    target_length = int(sr * duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode='constant')
    else:
        y = y[:target_length]
    return y

def safe_trim(y, sr, duration, top_db=25):
    try:
        y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
        if len(y_trimmed) > 0:
            y = y_trimmed
    except Exception:
        pass
    return pad_or_trim(y, sr, duration)

# Ekstraksi vektor fitur (sama seperti training)
def extract_fixed_vector_features(audio_path, sr=SR, duration=DURATION, target_dim=TARGET_VECTOR_DIM):
    y, sr = librosa.load(audio_path, sr=sr, duration=duration, mono=True)
    y = safe_trim(y, sr, duration)

    feats = []
    mfcc13 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    feats += list(np.mean(mfcc13, axis=1))
    feats += list(np.std(mfcc13, axis=1))

    mfcc20 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    feats += list(np.mean(mfcc20, axis=1))
    feats += list(np.std(mfcc20, axis=1))

    # Tambahkan fitur sederhana lain (contoh RMS, ZCR)
    rms = librosa.feature.rms(y=y)
    feats += [np.mean(rms), np.std(rms)]
    zcr = librosa.feature.zero_crossing_rate(y)
    feats += [np.mean(zcr), np.std(zcr)]

    features_array = np.array(feats, dtype=np.float32)
    features_array = np.nan_to_num(features_array)

    # Padding/truncating ke target_dim
    if len(features_array) < target_dim:
        features_array = np.pad(features_array, (0, target_dim - len(features_array)), mode='constant')
    elif len(features_array) > target_dim:
        features_array = features_array[:target_dim]

    return features_array

# Ekstraksi spectrogram (sama seperti training)
def extract_fixed_spectrogram(audio_path, sr=SR, duration=DURATION, n_mels=N_MELS, target_frames=SPEC_TIME_FRAMES):
    y, sr = librosa.load(audio_path, sr=sr, duration=duration, mono=True)
    y = safe_trim(y, sr, duration)

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    T = mel_db.shape[1]
    if T < target_frames:
        mel_db = np.pad(mel_db, ((0, 0), (0, target_frames - T)), mode='constant')
    elif T > target_frames:
        mel_db = mel_db[:, :target_frames]

    return mel_db.astype(np.float32)

# Fungsi test 1 file
def test_audio_file(file_path):
    print(f"🔍 Testing file: {file_path}")

    # Ekstrak fitur
    vec = extract_fixed_vector_features(file_path)
    spec = extract_fixed_spectrogram(file_path)

    # DNN prediction
    vec_scaled = scaler.transform([vec])
    dnn_pred = np.argmax(dnn_model.predict(vec_scaled, verbose=0), axis=1)[0]
    dnn_class = class_names[dnn_pred]

    # CNN prediction
    spec_input = spec[np.newaxis, ..., np.newaxis]
    cnn_pred = np.argmax(cnn_model.predict(spec_input, verbose=0), axis=1)[0]
    cnn_class = class_names[cnn_pred]

    print(f"🎯 DNN Prediction: {dnn_class}")
    print(f"🎯 CNN Prediction: {cnn_class}")

    return dnn_class, cnn_class

# ==== Contoh pemakaian ====
# Ganti dengan path file audio yang ingin diuji
file_path = r"D:\web\cnn_clasification\archive\Data\genres_original\audiobajingan\bajingan10.wav"
test_audio_file(file_path)




🔍 Testing file: D:\web\cnn_clasification\archive\Data\genres_original\audiobajingan\bajingan10.wav
🎯 DNN Prediction: audiobajingan
🎯 CNN Prediction: audiobajingan


('audiobajingan', 'audiobajingan')